## 1. Data loading and initial inspection


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

sns.set_theme(style="whitegrid")

os.makedirs("figures", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

df = pd.read_csv("data/raw/StudentPerformanceFactors.csv")
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")


In [ ]:
# Create a small DataFrame and Series by hand to satisfy the project requirements
exam_info = pd.DataFrame({
    "Exam_Period": ["Midterm", "Final"],
    "Date_String": ["2023-10-15", "2023-12-10"]
})

study_level_labels = pd.Series({
    "Normal": "Fewer than 20 study hours per week",
    "High": "20 or more study hours per week"
}, name="Study_Level_Definition")

display(exam_info)
display(study_level_labels)

exam_info["Date"] = pd.to_datetime(exam_info["Date_String"], errors="coerce")
display(exam_info)


In [ ]:
display(df.iloc[:3, :5])


In [ ]:
df.info()


In [ ]:
display(df.head())


## 2. Cleaning and transformation


In [ ]:
# Fill missing data with the constant "Unknown"
df["Teacher_Quality"] = df["Teacher_Quality"].fillna("Unknown")

# Drop rows where important context fields are missing
df = df.dropna(subset=["Parental_Education_Level", "Distance_from_Home"]).copy()

# Force "Previous_Scores" to become numeric
df["Previous_Scores"] = pd.to_numeric(df["Previous_Scores"], errors="coerce")

# Convert a categorical field to category dtype
df["Parental_Involvement"] = df["Parental_Involvement"].astype("category")

# Filter for students in public schools with high attendance
public_high_school_attendance = df[(df["School_Type"] == "Public") & (df["Attendance"] > 90)]
display(public_high_school_attendance.head())


In [ ]:
# Use .loc to create a new flag based on a condition
df.loc[df["Hours_Studied"] >= 20, "Study_Commitment"] = "High"
df.loc[df["Hours_Studied"] < 20, "Study_Commitment"] = "Normal"

# Categorize attendance using np.where
df["Attendance_Status"] = np.where(df["Attendance"] > 80, "Good", "Poor")

# Use .apply to create a custom performance band
def score_band(score):
    if score >= 80:
        return "Excellent"
    if score >= 70:
        return "Strong"
    if score >= 60:
        return "Developing"
    return "Needs Support"

df["Performance_Band"] = df["Exam_Score"].apply(score_band)

# Grouping and multi-column aggregation
grouped = df.groupby(["Parental_Education_Level", "Gender"])["Exam_Score"].agg(["count", "mean"]).reset_index()

display(df[["Exam_Score", "Performance_Band", "Study_Commitment", "Attendance_Status"]].head())
display(grouped)


In [ ]:
# Turn the grouped summary into a pivot table
pivot_df = grouped.pivot_table(index="Parental_Education_Level", columns="Gender", values="mean")
print("Mean exam score by parental education level and gender")
display(pivot_df)

# Export intermediate cleaned results
df.to_csv("data/processed/Cleaned_StudentPerformanceFactors.csv", index=False)


## 3. Descriptive statistics and simple EDA


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
display(df[numeric_cols].describe())


In [ ]:
# Explicit summary statistics for key numeric variables
key_stats = (
    df[["Hours_Studied", "Attendance", "Previous_Scores", "Exam_Score"]]
    .agg(["mean", "median", "min", "max", "std"])
    .T
    .round(2)
)
display(key_stats)


In [ ]:
# Descriptive stats by group for categorical variables
resource_summary = df.groupby("Access_to_Resources")["Exam_Score"].agg(["count", "mean"]).sort_values("mean", ascending=False)
parent_education_summary = df.groupby("Parental_Education_Level")["Exam_Score"].agg(["count", "mean"]).sort_values("mean", ascending=False)

display(resource_summary)
display(parent_education_summary)


In [ ]:
# Correlation matrix
corr_matrix = df[numeric_cols].corr()
display(corr_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", ax=ax)
ax.set_title("Correlation Matrix of Numeric Features")
fig.tight_layout()
fig.savefig("figures/correlation_heatmap.png", dpi=300)
plt.show()


## 4. Visualizations and feature exploration


### Research question 1
Do students who study more hours tend to earn higher exam scores?


In [ ]:
study_band_order = ["1-10", "11-20", "21-30", "31-40", "41-50"]
df["Study_Hour_Band"] = pd.cut(
    df["Hours_Studied"],
    bins=[0, 10, 20, 30, 40, 50],
    labels=study_band_order
)

study_band_summary = (
    df.groupby("Study_Hour_Band", observed=False)["Exam_Score"]
    .agg(["count", "mean"])
    .reset_index()
)

display(study_band_summary)

fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(
    data=study_band_summary,
    x="Study_Hour_Band",
    y="mean",
    marker="o",
    linewidth=2.5,
    color="#1d3557",
    ax=ax
)
ax.set_title("Average Exam Score by Weekly Study-Hour Band")
ax.set_xlabel("Study-Hour Band")
ax.set_ylabel("Average Exam Score")
fig.tight_layout()
fig.savefig("figures/line_study_hours_vs_exam_score.png", dpi=300)
plt.show()


Average exam scores rise as students move into higher study-hour bands. The line is not perfectly smooth, but the overall pattern supports the idea that more study time is associated with better performance.


### Research question 2
Do students with better access to educational resources perform better on average?


In [ ]:
resource_plot_data = (
    df.groupby("Access_to_Resources")["Exam_Score"]
    .agg(["count", "mean"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

display(resource_plot_data)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(
    data=resource_plot_data,
    x="Access_to_Resources",
    y="mean",
    hue="Access_to_Resources",
    dodge=False,
    palette=["#1d3557", "#457b9d", "#a8dadc"],
    ax=ax
)
if ax.legend_ is not None:
    ax.legend_.remove()
ax.set_title("Average Exam Score by Access to Resources")
ax.set_xlabel("Access to Resources")
ax.set_ylabel("Average Exam Score")
fig.tight_layout()
fig.savefig("figures/bar_resource_access_vs_exam_score.png", dpi=300)
plt.show()


Students in the high-resource group have the strongest average scores, while the low-resource group has the weakest. The difference is modest, but it is consistent enough to support a clear story about academic support and opportunity.


### Distribution of exam scores
How are exam scores distributed across the full dataset?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["Exam_Score"], bins=20, kde=True, color="#2a9d8f", ax=ax)
ax.set_title("Distribution of Exam Scores")
ax.set_xlabel("Exam Score")
ax.set_ylabel("Number of Students")
fig.tight_layout()
fig.savefig("figures/hist_exam_score_distribution.png", dpi=300)
plt.show()


Most scores cluster in the mid-to-upper 60s, with fewer students at the extreme low and high ends. This suggests the dataset is centered around a fairly typical performance range rather than being heavily skewed toward top scores.


### Attendance and exam performance
Is attendance associated with higher exam scores?


In [ ]:
scatter_sample = df.sample(n=min(1500, len(df)), random_state=42)
attendance_exam_corr = df[["Attendance", "Exam_Score"]].corr().iloc[0, 1]
print(f"Correlation between attendance and exam score: {attendance_exam_corr:.2f}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=scatter_sample,
    x="Attendance",
    y="Exam_Score",
    hue="Access_to_Resources",
    alpha=0.55,
    s=45,
    ax=ax
)
ax.set_title("Attendance vs. Exam Score")
ax.set_xlabel("Attendance")
ax.set_ylabel("Exam Score")
fig.tight_layout()
fig.savefig("figures/scatter_attendance_vs_exam_score.png", dpi=300)
plt.show()


The scatter plot shows an upward trend: students with stronger attendance generally earn higher exam scores. Attendance also has the strongest positive correlation with exam score in the correlation matrix, making it one of the most important variables in this exploratory analysis.


## 5. Summary and key findings


This analysis suggests that study habits and attendance are the strongest academic factors linked to exam performance in this dataset. Students who study more and attend more classes tend to have higher average scores, and attendance shows the strongest numeric relationship with exam score in the correlation matrix.

Support-related variables matter too. Students with high access to educational resources have better average outcomes than students with low access, and students whose parents have postgraduate education also perform slightly better on average than students in the other parental education groups.

Overall, the data story is that performance appears to improve when students combine consistent study time, strong attendance, and better academic support. Because this is an exploratory project, these findings describe patterns in the data rather than proving cause and effect.
